# 13 - function call

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
intructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

In [4]:
from rag_helper import RAGBase

In [5]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=intructions
)

In [6]:
assistant.rag('How do I run Ollama locally?')

'To run Ollama locally:\n\n1. **Install Ollama** from: https://ollama.com/download  \n   - **macOS**: download the `.pkg`\n   - **Windows**: download the `.msi`\n   - **Linux**: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start Ollama locally** by opening a terminal and running:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. **Check that the local server is running** with:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf you get a connection issue while prompting Ollama, restart the server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [7]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you mean **can you still join now**, check:\n- whether the course is **currently open for enrollment**\n- if there’s a **deadline** or waitlist\n- whether it’s **self-paced** or has a fixed start date\n- whether you meet any **prerequisites**\n\nIf you want, I can help you write a quick message to the instructor or course admin asking to join.'

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join the course discovered late can I join after course start enrollment late registration"}', call_id='call_xVQ4QaasWwxCRydlqRbQQV6c', name='search', type='function_call', id='fc_0052601319faa6a4006a3860a69f0081a186c58ce9482dfbb9', namespace=None, status='completed')]

In [11]:
call = response.output[0]

In [12]:
call

ResponseFunctionToolCall(arguments='{"query":"join the course discovered late can I join after course start enrollment late registration"}', call_id='call_xVQ4QaasWwxCRydlqRbQQV6c', name='search', type='function_call', id='fc_0052601319faa6a4006a3860a69f0081a186c58ce9482dfbb9', namespace=None, status='completed')

In [14]:
import json 
call.arguments

'{"query":"join the course discovered late can I join after course start enrollment late registration"}'

In [15]:
args = json.loads(call.arguments)

In [16]:
results = search(**args)

In [19]:
result_json = json.dumps(results, indent=2)

In [20]:
result_json

'[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "04919992b3",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm

In [22]:
function_call_output = {
    "type":"function_call_output",
    "call_id": call.call_id,
    "output": result_json
}

In [23]:
messages.append(call)
messages.append(function_call_output)

In [24]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [29]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, you’ll need to submit your project while submissions are still open.


In [33]:
response.usage.input_tokens
response.usage.output_tokens

32

# 14 - agentic loop

In [35]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [47]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

In [48]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

''

In [49]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment registration late join course FAQ"}', call_id='call_QKHyUhfDAUMMuwrNibDO6XTS', name='search', type='function_call', id='fc_094a46361c98f375006a3865cfbcb88191906cb432e3e8f2ad', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment open registration late add drop FAQ"}', call_id='call_ZlqwGQhLptZRaEZOvyA4SFby', name='search', type='function_call', id='fc_094a46361c98f375006a3865cfbcc881918d0c82087283581f', namespace=None, status='completed')]

In [50]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [52]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    
    if item.type == 'function_call':
        print("function_call:", item.name, item.arguments)

        call_output = make_call(call)
        messages.append(call_output)

        has_function_calls = True
        
    elif item.type == 'message':
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered can I join enrollment registration late join course FAQ"}
function_call: search {"query":"course enrollment open registration late add drop FAQ"}


In [53]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment late registration course FAQ"}
function_call: search {"query":"course enrollment join after start discovered the course FAQ"}


In [44]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late join FAQ"}', call_id='call_8UTdLqeesO1BVakWyCEA2ys2', name='search', type='function_call', id='fc_07efeb0d57941243006a38642557fc81a18e4d9f4c7293d863', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course discovered can I join now enrollment FAQ"}', call_id='call_YRZn11AU7N1Y

In [55]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join and start learning anytime.

A couple of important notes:
- You can go through the materials at your own pace.
- If you want a certificate, you need to submit your project while submissions are still open.

If you want, I can also help you with the best way to start the course or explain the certificate rules.


In [56]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [57]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run Ollama locally install run local FAQ"}
function_call: search {"query":"Ollama locally run model install FAQ"}
function_call: search {"query":"Olama typo Ollama local setup course FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - Go to: https://ollama.com/download
   - Choose your OS:
     - **macOS**: install the `.pkg`
     - **Windows**: install the `.msi`
     - **Linux**:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - Go to: https://ollama.com/download\n   - Choose your OS:\n     - **macOS**: install the `.pkg`\n     - **Windows**: install the `.msi`\n     - **Linux**:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nnohup ollama serve > nohup.o

In [58]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
ASSISTANT:
I couldn’t find a relevant course FAQ entry for “queen gambit,” so I can’t answer that from the course materials.

If you meant something course-related, could you rephrase it with more context? Are there other areas you want to explore?


'I couldn’t find a relevant course FAQ entry for “queen gambit,” so I can’t answer that from the course materials.\n\nIf you meant something course-related, could you rephrase it with more context? Are there other areas you want to explore?'

In [54]:
response.output_text

''

# 15 - frameworks

In [61]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [62]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [63]:
search_tool

{'type': 'function',
 'name': 'search',
 'description': 'Search the FAQ database for entries matching the given query.',
 'parameters': {'type': 'object',
  'properties': {'query': {'type': 'string',
    'description': 'Search query text to look up in the course FAQ.'}},
  'required': ['query'],
  'additionalProperties': False}}

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [66]:
agent_tools.get_tools() # you don't need to define the agent tool by yourself, the frameworks usually already define them

[{'type': 'function',
  'name': 'search',
  'description': 'No description provided.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [67]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [68]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [69]:
result.cost

CostInfo(input_cost=Decimal('0.00104175'), output_cost=Decimal('0.0010395'), total_cost=Decimal('0.00208125'))